In [5]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import os
import concurrent.futures
import re
from collections import Counter
import numpy as np
import hashlib

from dotenv import load_dotenv

# Загрузка переменных окружения
load_dotenv()
# Подключаем ключ для LLM-модели
LLM_API_KEY = os.getenv("OPENAI_API_KEY")
# Подключаем ключ для EMBEDDER-модели
EMBEDDER_API_KEY = os.getenv("EMBEDDER_API_KEY")

# Модели с весами
LLM_MODELS = {
    "openrouter/google/gemma-3-27b-it": 1.5,
}

def load_train_data():
    """Загрузка тренировочных данных"""
    try:
        if os.path.exists('./train_data.csv'):
            train_df = pd.read_csv('./train_data.csv')
            if 'text' in train_df.columns:
                print(f"Загружено {len(train_df)} строк тренировочных данных")
                return train_df
        else:
            print("Файл train_data.csv не найден")
            return pd.DataFrame()
        
    except Exception as e:
        print(f"Ошибка при загрузке train_data.csv: {e}")
        return pd.DataFrame()

def precompute_text_vectors(train_df):
    """Предварительное вычисление векторных представлений текстов"""
    if train_df.empty or 'text' not in train_df.columns:
        return {}
    
    text_vectors = {}
    for idx, row in train_df.iterrows():
        text = str(row['text']).lower()
        words = set(re.findall(r'\w+', text))
        text_vectors[idx] = words
    
    print(f"Предварительно обработано {len(text_vectors)} текстов")
    return text_vectors

def find_relevant_context_fast(question, train_df, text_vectors):
    """Быстрый поиск релевантного контекста"""
    if train_df.empty or not text_vectors:
        return ""
    
    question_lower = question.lower()
    question_words = set(re.findall(r'\w+', question_lower))
    
    relevant_scores = []
    for idx, text_words in text_vectors.items():
        common_words = question_words.intersection(text_words)
        score = len(common_words)
        if score >= 2:
            relevant_scores.append((idx, score))
    
    # Сортируем по релевантности и берем топ-3
    relevant_scores.sort(key=lambda x: x[1], reverse=True)
    top_indices = [idx for idx, _ in relevant_scores[:3]]
    
    if not top_indices:
        return "\n\n".join(train_df['text'].head(3).tolist())
    
    return "\n\n".join([train_df.iloc[idx]['text'] for idx in top_indices])

def is_russian(text, threshold=0.6):
    """Проверка, является ли текст русским"""
    if not text or not isinstance(text, str):
        return False
    
    russian_chars = len(re.findall(r'[а-яА-ЯёЁ]', text))
    total_chars = len(re.findall(r'[a-zA-Zа-яА-ЯёЁ]', text))
    
    if total_chars == 0:
        return False
    
    return (russian_chars / total_chars) >= threshold

def translate_to_russian(text, client, model):
    """Перевод текста на русский язык"""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": f"Переведи следующий текст на русский язык. Сохрани смысл и технические термины, но используй только русский язык:\n\n{text}"}
                ]
            }],
            temperature=0.1,
            max_tokens=1000
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Ошибка перевода: {e}")
        return text

def ensure_russian_answer(answer, client, model):
    """Гарантия, что ответ на русском языке"""
    if is_russian(answer):
        return answer
    
    russian_answer = translate_to_russian(answer, client, model)
    
    if not is_russian(russian_answer):
        return "Извините, не могу предоставить ответ на русском языке. Пожалуйста, переформулируйте вопрос."
    
    return russian_answer

def clean_response(text):
    """Удаляет лишние фразы из ответа"""
    if not text:
        return text
    
    # Фразы для удаления
    unwanted_phrases = [
        r'ответа не содержится в базе данных[^.]*\.?\s*',
        r'информация не найдена в базе знаний[^.]*\.?\s*',
        r'в предоставленной информации нет ответа[^.]*\.?\s*',
        r'база знаний не содержит[^.]*\.?\s*',
        r'но я попробую ответить сам[^.]*\.?\s*',
        r'но я могу ответить на основе[^.]*\.?\s*',
        r'однако я могу объяснить[^.]*\.?\s*',
        r'на основе своих знаний[^.]*\.?\s*',
        r'используя свои собственные знания[^.]*\.?\s*',
        r'в отсутствие информации из базы[^.]*\.?\s*',
    ]
    
    cleaned_text = text
    for phrase in unwanted_phrases:
        cleaned_text = re.sub(phrase, '', cleaned_text, flags=re.IGNORECASE)
    
    # Убираем двойные пробелы и лишние переносы
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
    cleaned_text = cleaned_text.strip()
    
    return cleaned_text

def generate_answer_for_model(args):
    """Функция для генерации ответа одной моделью"""
    question_idx, question, model, train_df, text_vectors, llm_api_key = args
    
    try:
        client = OpenAI(
            base_url="https://ai-for-finance-hack.up.railway.app/",
            api_key=llm_api_key,
        )
        
        context = find_relevant_context_fast(question, train_df, text_vectors)
        
        if context:
            prompt = f"""Ты - помощник, который отвечает на русском языке. Ответь на вопрос, используя информацию из базы знаний и при необходимости свои знания.

ИНФОРМАЦИЯ ИЗ БАЗЫ ЗНАНИЙ:
{context}

ВОПРОС: {question}

ВАЖНЫЕ ИНСТРУКЦИИ:
1. Отвечай ТОЛЬКО на русском языке
2. НЕ упоминай в ответе, что информация из базы знаний или твоих знаний
3. НЕ говори "в базе данных сказано", "согласно предоставленной информации" и т.п.
4. Просто дай подробный ответ на вопрос
5. Если информации недостаточно, дополни ответ своими знаниями БЕЗ упоминания об этом
6. Будь точным и информативным
7. Не используй английские слова без крайней необходимости
8. Отвечай непосредственно на заданный вопрос
"""

        else:
            prompt = f"""Ты - помощник, который отвечает на русском языке. Ответь на следующий вопрос:

ВОПРОС: {question}

ВАЖНЫЕ ИНСТРУКЦИИ:
1. Отвечай ТОЛЬКО на русском языке
2. Просто дай подробный ответ на вопрос
3. Не упоминай источники информации
4. Будь точным и информативным
5. Не используй английские слова без крайней необходимости
6. Отвечай непосредственно на заданный вопрос
"""

        response = client.chat.completions.create(
            model=model,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt}
                ]
            }],
            temperature=0.3,
            max_tokens=1500
        )
        answer = response.choices[0].message.content.strip()
        
        # Очищаем ответ от нежелательных фраз
        cleaned_answer = clean_response(answer)
        
        # Проверяем и переводим на русский при необходимости
        russian_answer = ensure_russian_answer(cleaned_answer, client, model)
        return question_idx, model, russian_answer, True
        
    except Exception as e:
        error_msg = f"Ошибка при генерации ответа: {str(e)}"
        print(f"Ошибка в модели {model} для вопроса {question_idx}: {e}")
        return question_idx, model, error_msg, False

def calculate_similarity(answer1, answer2):
    """Вычисляет схожесть между двумя ответами"""
    if not answer1 or not answer2:
        return 0.0
    
    words1 = set(re.findall(r'\w+', answer1.lower()))
    words2 = set(re.findall(r'\w+', answer2.lower()))
    
    if not words1 or not words2:
        return 0.0
    
    intersection = len(words1.intersection(words2))
    union = len(words1.union(words2))
    
    return intersection / union if union > 0 else 0.0

def analyze_answer_quality(answer):
    """Анализирует качество ответа для дополнительного взвешивания"""
    if not answer or not isinstance(answer, str):
        return 0.5
    
    quality_score = 0.5  # Базовый балл
    
    # Бонусы за качественные характеристики
    if len(answer) > 100:  # Развернутый ответ
        quality_score += 0.2
    if re.search(r'\d+\.\s|\n-|\n•', answer):  # Структурированный ответ
        quality_score += 0.15
    if re.search(r'(пример|например|к примеру|допустим)', answer.lower()):  # Есть примеры
        quality_score += 0.1
    if len(re.findall(r'[.!?]', answer)) >= 2:  # Хорошая пунктуация
        quality_score += 0.05
    
    return min(quality_score, 1.0)

def weighted_voting(answers_with_weights, similarity_threshold=0.3):
    """
    Упрощенное взвешенное голосование между ответами моделей
    """
    if not answers_with_weights:
        return "Не удалось сгенерировать ответ"
    
    # Если только один ответ, возвращаем его
    if len(answers_with_weights) == 1:
        return answers_with_weights[0][0]
    
    # Добавляем оценку качества к весам
    enhanced_answers = []
    for answer, base_weight in answers_with_weights:
        quality_score = analyze_answer_quality(answer)
        enhanced_weight = base_weight * (0.7 + 0.3 * quality_score)
        enhanced_answers.append((answer, enhanced_weight))
    
    # Простая стратегия: берем ответ с наибольшим весом
    best_answer, best_weight = max(enhanced_answers, key=lambda x: x[1])
    
    return best_answer

def process_questions_sequential(questions_list, train_df, text_vectors):
    """
    Надежная последовательная обработка вопросов с сохранением порядка
    """
    results = []
    
    for question_idx, question in enumerate(tqdm(questions_list, desc="Обработка вопросов")):
        print(f"\nОбрабатываю вопрос {question_idx+1}/{len(questions_list)}")
        print(f"Вопрос: {question[:100]}...")
        
        # Собираем ответы от всех моделей для текущего вопроса
        question_answers = []
        
        for model in LLM_MODELS.keys():
            args = (question_idx, question, model, train_df, text_vectors, LLM_API_KEY)
            try:
                _, model_name, answer, success = generate_answer_for_model(args)
                if success:
                    weight = LLM_MODELS[model]
                    question_answers.append((answer, weight))
                    print(f"  Модель {model}: получен ответ ({len(answer)} символов)")
                else:
                    print(f"  Модель {model}: ошибка")
            except Exception as e:
                print(f"  Модель {model}: исключение {e}")
                continue
        
        # Применяем взвешенное голосование
        if question_answers:
            # Фильтруем только русские ответы
            russian_answers = [(answer, weight) for answer, weight in question_answers if is_russian(answer)]
            
            if russian_answers:
                consensus_answer = weighted_voting(russian_answers)
            else:
                # Если нет русских ответов, переводим лучший
                best_answer = max(question_answers, key=lambda x: x[1])[0]
                client = OpenAI(
                    base_url="https://ai-for-finance-hack.up.railway.app/",
                    api_key=LLM_API_KEY,
                )
                consensus_answer = translate_to_russian(best_answer, client, list(LLM_MODELS.keys())[0])
            
            results.append(consensus_answer)
        else:
            # Резервный ответ
            default_answer = "На основе имеющейся информации можно сказать, что данный вопрос требует дополнительного изучения. Рекомендуется обратиться к официальным источникам для получения наиболее точной и актуальной информации."
            results.append(default_answer)
    
    return results

def validate_question_answer_pairs(questions_list, answers_list):
    
    if len(questions_list) != len(answers_list):
        return False
    
    # Проверяем релевантность первых нескольких пар
    for i in range(min(2, len(questions_list))):
        question_words = set(re.findall(r'\w+', questions_list[i].lower()))
        answer_words = set(re.findall(r'\w+', answers_list[i].lower()))
        common_words = question_words.intersection(answer_words)
    
    return True

def main():
    
    # Загрузка данных
    train_df = load_train_data()
    text_vectors = precompute_text_vectors(train_df)
    
    # Загрузка вопросов
    try:
        questions_df = pd.read_csv('./questions.csv')
        questions_list = questions_df['Вопрос'].tolist()
        return
    
    # Обработка вопросов
    print("\n=== НАЧАЛО ОБРАБОТКИ ===")
    final_answers = process_questions_sequential(questions_list, train_df, text_vectors)
    
    # Валидация результатов
    print("\n=== ПРОВЕРКА РЕЗУЛЬТАТОВ ===")
    validation_passed = validate_question_answer_pairs(questions_list, final_answers)
    
    if not validation_passed:
        print("Обнаружены проблемы с валидацией!")
        return
    
    # Создание финального DataFrame
    result_df = pd.DataFrame({
        'Вопрос': questions_list,
        'Консенсус-ответ': final_answers
    })
    
    # Сохранение результатов
    output_file = 'submission_final.csv'
    result_df.to_csv(output_file, index=False, encoding='utf-8-sig')

if __name__ == "__main__":
    main()

=== ЗАПУСК ОБРАБОТКИ ВОПРОСОВ ===

Загружено 350 строк тренировочных данных
Предварительно обработано 350 текстов
Загружено 500 вопросов

=== НАЧАЛО ОБРАБОТКИ ===


Обработка вопросов: 100%|█████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 68985.26it/s]


Обрабатываю вопрос 1/500
Вопрос: Как просрочка по «беспроцентному» займу скажется на переплате/ПСК?...
Ошибка в модели openrouter/google/gemma-3-27b-it для вопроса 0: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
  Модель openrouter/google/gemma-3-27b-it: ошибка
  Использован резервный ответ

Обрабатываю вопрос 2/500
Вопрос: Как действовать вкладчику при отзыве лицензии, учитывая лимит безопасной суммы?...
Ошибка в модели openrouter/google/gemma-3-27b-it для вопроса 1: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
  Модель openrouter/google/gemma-3-27b-it: ошибка
  Использован резервный ответ

Обрабатываю вопрос 3/500
Вопрос: Как действовать при оспаривании незаконной продажи долга с учетом установленных сроков ответа на пре...
Ошибка в модели openrouter/google/gemma-3-27b-it для вопроса 2: The api_key client option